In [ ]:
import ee, eemont
import geemap
import geemap.colormaps as geecm

In [ ]:
import operator
import numpy as np
import pandas as pd

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
# Import need to be done after initialization to get Classifier
from pygee import gee
from pygee.tools.lc_mapping import LandCoverMap

# 1. General parameters

Scale and projection

In [ ]:
scale = 20
projection = "EPSG:3035"

Nomenclature

In [ ]:
pygee_dir = "to_complete_with_pygee_install_dir/"

In [ ]:
mapping_file = os.path.join(pygee_dir, "tools/nomenclature_h1b.txt"
mapping_name = "h1b_paper"

In [ ]:
lcmap = LandCoverMap(mapping_file, name=mapping_name)
lcmap_classify = lcmap.remove_item(col_name="Code", col_val=[4,5,6,8,9], in_place=False)

Savings

In [ ]:
save_dir = "to-complete-with-local-dir/"

# 2. Datasets

LIA shape

In [ ]:
fc_lia = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_with_buffer200')
geom_lia = fc_lia.geometry().convexHull()

LIA indices

In [ ]:
s2_indices = ee.Image("users/aguerou/ice_and_life/carto_h1b/indices/s2_indices_nari_ncri_median_ts_lia_Reinthaler_Mourey_with_buffer200")

Training dataset

In [ ]:
ds_training = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/dataset_training/ds_training_g25_nari_ncri_median_ts')

RF areas

In [ ]:
rf_areas = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/data_ancillary/rf_areas')

# 3. Random Forest - classification LIA

In [ ]:
indices_ts = ["NARI_7", "NCRI_7", 
              "NARI_8", "NCRI_8", 
              "NARI_9", "NCRI_9"]

## 3.1 Get RF parameters

In [ ]:
training_label = "landcover"
variablesPerSplit = 3 #default to sqrt(6)=2.4
minLeafPopulation = 1 

In [ ]:
obb_res = []
n_trees = [1,5,10,20,30,50,80,100,120,150,180,200,300,400,500]

for n in n_trees:
    rf_gee = ee.Classifier.smileRandomForest(n, variablesPerSplit=variablesPerSplit, minLeafPopulation=minLeafPopulation)
    classifier = rf_gee.train(ds_training, training_label, indices_ts)
    obb_res.append(classifier.explain().getInfo()["outOfBagErrorEstimate"])

In [ ]:
plt.figure()
plt.plot(n_trees, obb_res, marker="o")
plt.ylim([0.04,0.15])
plt.grid(ls="--")

## 3.2 Classify

In [ ]:
n_trees = 200
variablesPerSplit = 3 #default to sqrt(6)=2.4
minLeafPopulation = 1 
training_label = "landcover"

In [ ]:
# Declare the Random forest
rf_gee = ee.Classifier.smileRandomForest(n_trees, 
                                         variablesPerSplit=variablesPerSplit, 
                                         minLeafPopulation=minLeafPopulation)

In [ ]:
rf_mean_ts, rf_std_ts, errorMatrix_ts, scores = gee.rf_circular(rf_gee, 
                                                                ds_training, 
                                                                s2_indices, 
                                                                indices_names=indices_ts, 
                                                                labels_lc=list(lcmap_classify.get_type()),
                                                                n_areas=4,
                                                                areas_name="kfold_id")

### Define sparse vegetation

We define Sparse Veget as pixel with predictive set equal to {rocks ; grass} @ 95% confidence level, the level choosen to estimate LC uncertainties

So rocks and grass with proba > q_score(1-95%) and other classes with proba < q_score(1-95%)

In [ ]:
q_score = scores.quantile(q=0.05)
q_score

In [ ]:
classif_ts = gee.classify(rf_mean_ts, 
                          lc_labels=list(lcmap_classify.get_type()), 
                          lc_values=list(lcmap_classify.get_code()), 
                          extra_class=[(lcmap_classify.get_type()[0], operator.gt, q_score),
                                       (lcmap_classify.get_type()[1], operator.gt, q_score),
                                       (lcmap_classify.get_type()[2], operator.lt, q_score),
                                       (lcmap_classify.get_type()[3], operator.lt, q_score),
                                       ], 
                          extra_class_value=lcmap.get_code_of_type(type="sparse veget."))

## 3.3 Error Matrix

In [ ]:
gee.plot_errorMatrix(errorMatrix_ts[0], 
                     lcmap=lcmap_classify, 
                     label="Area 1", 
                     save_dir=save_dir , 
                     save_name="error_matrix_g25_median_ts_indices_rf1.png")

In [ ]:
gee.plot_errorMatrix(errorMatrix_ts[1], 
                     lcmap=lcmap_classify, 
                     label="Area 2", 
                     save_dir=save_dir , 
                     save_name="error_matrix_g25_median_ts_indices_rf2.png")

In [ ]:
gee.plot_errorMatrix(errorMatrix_ts[2], 
                     lcmap=lcmap_classify, 
                     label="Area 3", 
                     save_dir=save_dir , 
                     save_name="error_matrix_g25_median_ts_indices_rf3.png")

In [ ]:
gee.plot_errorMatrix(errorMatrix_ts[3], 
                     lcmap=lcmap_classify, 
                     label="Area 4", 
                     save_dir=save_dir, 
                     save_name="error_matrix_g25_median_ts_indices_rf4.png")

In [ ]:
e0_ts = np.array(errorMatrix_ts[0].getInfo())
e1_ts = np.array(errorMatrix_ts[1].getInfo())
e2_ts = np.array(errorMatrix_ts[2].getInfo())
e3_ts = np.array(errorMatrix_ts[3].getInfo())

In [ ]:
em_ts_glob = ee.ConfusionMatrix(ee.Array((e0_ts+e1_ts+e2_ts+e3_ts).tolist()))

In [ ]:
gee.plot_errorMatrix(em_ts_glob, 
                     lcmap=lcmap_classify, 
                     label="All areas", 
                     save_dir=save_dir, 
                     save_name="error_matrix_g25_median_ts_indices_all_rf.png")

### Save global matrix

In [ ]:
pd.DataFrame((e0_ts+e1_ts+e2_ts+e3_ts)).to_csv(os.path.join(save_dir, "confusion_matrix_g25.csv"))

# 4. Save as assets

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
Map.addLayer(fc_lia)
Map.addLayer(classif_ts.updateMask(classif_ts.gt(-1)), {"min":-1, "max":6, "palette":["#ffffff"]+list(lcmap.get_colors()[:-2])})
Map

## 4.1 Carto and proba

In [ ]:
geemap.ee_export_image_to_asset(classif_ts,
                                assetId="users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_rf_only",
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=geom_lia)

In [ ]:
geemap.ee_export_image_to_asset(rf_mean_ts.rename(["rocks"] + rf_mean_ts.bandNames()[1:]),
                                assetId="users/aguerou/ice_and_life/carto_h1b/carto/carto_h1b_lia_Reinthaler_Mourey_median_ts_g25_rf_only_proba",
                                scale=scale,
                                crs=projection,
                                maxPixels=3e10,
                                region=geom_lia)

## 4.2 Scores for uncertainty

In [ ]:
scores.to_csv(os.path.join(save_dir, "rf_validation_scores_g25.csv"))